# 🔌 EV Market Analysis 2026

**Author:** Yash Gupta — Business Analyst  
**Dataset:** `ev_market_2026.csv`  
**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn · Plotly

---

## Objective
Perform a comprehensive Exploratory Data Analysis (EDA) on the 2026 global EV market dataset to uncover pricing trends, performance benchmarks, market segmentation patterns, and geographic insights across major automotive brands.

---

### Table of Contents
1. [Import Libraries & Load Data](#1)
2. [Data Overview & Quality Check](#2)
3. [Brand Analysis](#3)
4. [Price Analysis](#4)
5. [Performance Metrics](#5)
6. [Market Segmentation](#6)
7. [Geographic Insights](#7)
8. [Correlation Analysis](#8)
9. [Technology & Autopilot](#9)
10. [Key Takeaways](#10)

---
## 1. Import Libraries & Load Data <a id='1'></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13, 'axes.labelsize': 11})

print('✅ Libraries loaded successfully!')

In [ ]:
df = pd.read_csv('data/ev_market_2026.csv')
print(f'Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

---
## 2. Data Overview & Quality Check <a id='2'></a>

In [ ]:
print('Shape:', df.shape)
print('\nData Types:')
print(df.dtypes)
print('\nMissing Values:')
print(df.isnull().sum())

In [ ]:
df.describe().round(2)

In [ ]:
print('Unique Brands:', df['brand'].nunique(), '->', df['brand'].unique().tolist())
print('Year Range:', df['year'].min(), '–', df['year'].max())
print('Countries:', df['country_of_origin'].unique().tolist())
print('Segments:', df['market_segment'].unique().tolist())
print('Drive Types:', df['drive_type'].unique().tolist())

---
## 3. Brand Analysis <a id='3'></a>

In [ ]:
brand_sales = df.groupby('brand')['annual_sales_units'].sum().sort_values(ascending=False).reset_index()
brand_price = df.groupby('brand')['price_usd'].mean().sort_values(ascending=False).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Brand-Level Analysis', fontsize=15, fontweight='bold')

# Top 10 brands by sales
sns.barplot(data=brand_sales.head(10), y='brand', x='annual_sales_units', palette='Blues_d', ax=axes[0])
axes[0].set_title('Top 10 Brands by Annual Sales')
axes[0].set_xlabel('Total Annual Sales')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

# Avg price by brand
sns.barplot(data=brand_price, y='brand', x='price_usd', palette='Oranges_d', ax=axes[1])
axes[1].set_title('Average Price by Brand (USD)')
axes[1].set_xlabel('Average Price')
axes[1].set_ylabel('')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

plt.tight_layout()
plt.show()

In [ ]:
# Interactive Plotly Chart — Sales by Brand
fig = px.bar(
    brand_sales.head(12), x='annual_sales_units', y='brand',
    orientation='h', color='annual_sales_units',
    color_continuous_scale='Blues',
    title='Annual EV Sales by Brand (Top 12)',
    labels={'annual_sales_units': 'Annual Sales', 'brand': 'Brand'}
)
fig.update_layout(height=480, coloraxis_showscale=False)
fig.show()

---
## 4. Price Analysis <a id='4'></a>

In [ ]:
seg_order = ['Budget', 'Mid-range', 'Premium', 'Luxury']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Price Analysis', fontsize=15, fontweight='bold')

# Price distribution
axes[0].hist(df['price_usd'], bins=40, color='#4C72B0', edgecolor='white', alpha=0.85)
axes[0].set_title('Price Distribution (All EVs)')
axes[0].set_xlabel('Price (USD)')
axes[0].set_ylabel('Count')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# Box plot by segment
sns.boxplot(data=df, x='market_segment', y='price_usd', order=seg_order, palette='Set2', ax=axes[1])
axes[1].set_title('Price Distribution by Market Segment')
axes[1].set_xlabel('Segment')
axes[1].set_ylabel('Price (USD)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

plt.tight_layout()
plt.show()

In [ ]:
# Price by country of origin
fig = px.violin(
    df, x='country_of_origin', y='price_usd', color='country_of_origin',
    box=True, points='outliers',
    title='Price Distribution by Country of Origin',
    labels={'price_usd': 'Price (USD)', 'country_of_origin': 'Country'}
)
fig.update_layout(showlegend=False)
fig.show()

---
## 5. Performance Metrics <a id='5'></a>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Performance Metrics', fontsize=15, fontweight='bold')

# Range vs Price scatter
sc = axes[0].scatter(
    df['range_miles'], df['price_usd'],
    c=df['battery_capacity_kwh'], cmap='viridis',
    alpha=0.6, edgecolors='none', s=25
)
plt.colorbar(sc, ax=axes[0], label='Battery Capacity (kWh)')
axes[0].set_title('Range vs Price\n(color = battery capacity)')
axes[0].set_xlabel('Range (miles)')
axes[0].set_ylabel('Price (USD)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# Horsepower by segment
sns.violinplot(data=df, x='market_segment', y='horsepower', order=seg_order, palette='coolwarm', ax=axes[1])
axes[1].set_title('Horsepower by Market Segment')
axes[1].set_xlabel('Segment')
axes[1].set_ylabel('Horsepower (hp)')

plt.tight_layout()
plt.show()

In [ ]:
# Interactive: Acceleration vs Range by brand
fig = px.scatter(
    df, x='range_miles', y='acceleration_0_60_mph',
    color='brand', size='annual_sales_units',
    hover_data=['model', 'price_usd', 'market_segment'],
    title='Range vs Acceleration (bubble size = annual sales)',
    labels={'range_miles': 'Range (miles)', 'acceleration_0_60_mph': '0–60 mph (sec)'}
)
fig.update_layout(height=500)
fig.show()

---
## 6. Market Segmentation <a id='6'></a>

In [ ]:
seg_counts = df['market_segment'].value_counts().reindex(seg_order)
seg_sales = df.groupby('market_segment')['annual_sales_units'].sum().reindex(seg_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Market Segmentation', fontsize=15, fontweight='bold')

colors = ['#2ecc71', '#3498db', '#9b59b6', '#e74c3c']
axes[0].pie(seg_counts, labels=seg_order, autopct='%1.1f%%', colors=colors, startangle=140)
axes[0].set_title('Model Listings by Segment')

axes[1].pie(seg_sales, labels=seg_order, autopct='%1.1f%%', colors=colors, startangle=140)
axes[1].set_title('Annual Sales Volume by Segment')

plt.tight_layout()
plt.show()

In [ ]:
# Brand presence across segments
brand_seg = df.groupby(['brand', 'market_segment']).size().unstack(fill_value=0)
brand_seg_pct = brand_seg.div(brand_seg.sum(axis=1), axis=0).round(2)

fig = px.bar(
    brand_seg_pct.reset_index().melt(id_vars='brand'),
    x='brand', y='value', color='market_segment',
    title='Brand Positioning Across Market Segments (% of listings)',
    labels={'value': '% of Listings', 'market_segment': 'Segment'},
    color_discrete_sequence=['#2ecc71', '#3498db', '#9b59b6', '#e74c3c']
)
fig.update_layout(xaxis_tickangle=-30, height=480)
fig.show()

---
## 7. Geographic Insights <a id='7'></a>

In [ ]:
country_sales = df.groupby('country_of_origin')['annual_sales_units'].sum().sort_values(ascending=False).reset_index()

fig = px.bar(
    country_sales, x='country_of_origin', y='annual_sales_units',
    color='annual_sales_units', color_continuous_scale='Teal',
    title='Total Annual Sales by Country of Origin',
    labels={'annual_sales_units': 'Annual Sales', 'country_of_origin': 'Country'}
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [ ]:
# Average price & range by country
country_metrics = df.groupby('country_of_origin')[['price_usd', 'range_miles', 'customer_rating']].mean().round(1)
print('Average Metrics by Country of Origin:')
country_metrics

---
## 8. Correlation Analysis <a id='8'></a>

In [ ]:
numeric_cols = [
    'price_usd', 'battery_capacity_kwh', 'range_miles', 'charging_speed_kw',
    'acceleration_0_60_mph', 'horsepower', 'torque_nm',
    'safety_rating', 'autopilot_level', 'annual_sales_units', 'customer_rating'
]

corr = df[numeric_cols].corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, linewidths=0.5,
    annot_kws={'size': 8}
)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print('Top Correlations with Price:')
corr['price_usd'].sort_values(ascending=False).drop('price_usd')

---
## 9. Technology & Autopilot <a id='9'></a>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Technology Insights', fontsize=15, fontweight='bold')

autopilot_dist = df['autopilot_level'].value_counts().sort_index()
axes[0].bar([f'Level {i}' for i in autopilot_dist.index], autopilot_dist.values,
            color=['#95a5a6', '#3498db', '#2ecc71', '#e74c3c'])
axes[0].set_title('SAE Autopilot Level Distribution')
axes[0].set_xlabel('Autopilot Level')
axes[0].set_ylabel('Number of Models')

drive_dist = df['drive_type'].value_counts()
axes[1].pie(drive_dist.values, labels=drive_dist.index, autopct='%1.1f%%',
            colors=['#3498db', '#e67e22', '#2ecc71'], startangle=90)
axes[1].set_title('Drive Type Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Autopilot level by brand
autopilot_brand = df.groupby('brand')['autopilot_level'].mean().sort_values(ascending=False).round(2)
print('Average Autopilot Level by Brand:')
autopilot_brand

---
## 10. Key Takeaways <a id='10'></a>

### 🏆 Market Leaders
- **Tesla** dominates global EV sales volume, particularly in the Luxury segment. The Model Y consistently exceeds **400,000+ units/year**.
- **BYD** is the volume champion in Budget and Mid-range segments, with the Seagull starting at just **~$16,000** — a strategic entry into mass-market EV adoption.

### 💰 Pricing Dynamics
- The EV market spans an enormous price range — from **$16K (BYD Seagull)** to over **$210K (Mercedes EQG)**.
- The **Mid-range segment ($40K–$80K)** has the highest number of listings, signaling market maturation.
- **US and Germany** command the highest average vehicle prices, while **China** leads in budget-accessible options.

### ⚡ Performance
- **Battery capacity** and **horsepower** are the strongest predictors of price.
- **AWD** drivetrain is heavily associated with Performance variants across all brands.
- Top performance models hit 0–60 mph in under **3 seconds**.

### 🤖 Technology
- **SAE Level 2** autopilot is the most common across the market.
- **Level 3** autonomy appears exclusively in select Tesla models in this dataset.
- Most budget models offer **Level 0** (no autopilot), highlighting the tech-price gap.

### 🌍 Geography
- **US** and **China** lead in annual EV sales volume.
- **South Korea** (Hyundai, Kia) punches above its weight with strong Premium segment offerings.
- **Germany** (BMW, Mercedes, Volkswagen, Audi) commands the Luxury end with premium engineering.

---

> **Author:** Yash Gupta | Business Analyst  
> 📧 yashguptayg9013.ie@gmail.com  
> 🔗 [LinkedIn](https://www.linkedin.com/in/YOUR_LINKEDIN) · [GitHub](https://github.com/YOUR_USERNAME)